# Benchmark Suite - Real SNOMED Methods

This notebook runs all benchmarks using actual implementations from the repository. Configure paths to your UK SNOMED CT data below.

In [ ]:
%matplotlib inline
from IPython.display import display

# Import evaluation functions
from snomed_methods.benchmarking.annotation import (
    evaluate_annotator,
    generate_annotation_dataset,
)

In [ ]:
try:
    from snomed_methods import (
        ClinicalConceptAnnotator,
        ConceptMapper,
        SnomedRelations,
        SnomedTermLookup,
    )

    HAS_METHODS = True
except ImportError as e:
    print(f"Real SNOMED methods not available: {e}")
    HAS_METHODS = False

## Define Real Methods

Update these paths to your actual SNOMED data locations:

In [ ]:
def real_annotator(text: str):
    uk_path = "/path/to/uk_sct2cl_42.2.0"
    model_path = None
    annotator = ClinicalConceptAnnotator(uk_path=uk_path, model_path=model_path)
    result = annotator.annotate(text, top_k=10)
    return [c.concept_id for c in result.top_concepts]


def real_term_lookup(term: str):
    description_path = "/path/to/snomed_description.txt"
    lookup = SnomedTermLookup(snomed_description_path=description_path)
    return lookup.find_concepts_by_term(term=term, top_n=20)


def real_mapper(snomed_cui: str):
    uk_path = "/path/to/uk_sct2cl_42.2.0"
    mapper = ConceptMapper(uk_path=uk_path)
    icd_mappings = mapper.load_icd_mapping()
    return icd_mappings.get(snomed_cui, [])[:5]


def real_hierarchy_expansion(seed_cui: str):
    rf2_path = "/path/to/uk_sct2cl_42.2.0"
    relations = SnomedRelations(snomed_rf2_full_path=rf2_path)
    children = relations.get_children(seed_cui)
    parents = relations.get_parents(seed_cui)
    return children[:10] + parents[:5]

## Run Annotation Benchmark

In [ ]:
if HAS_METHODS:
    dataset = generate_annotation_dataset(50)
    result = evaluate_annotator(real_annotator, dataset[:20], k_values=[1, 3, 5])
    display(result)
else:
    print("Configure UK data path and rerun")

## Next Steps

1. Configure UK SNOMED RF2 paths in the real method definitions above
2. Run full benchmarks with real data for meaningful metrics